In [17]:
from typing import Dict, List, Union

from pydantic import BaseModel
import requests

class URLParams(BaseModel):
    latitude: float
    longitude: float
    start_date: str
    end_date: str
    hourly: Union[str, List[str]]
    timezone: str


api_data_url="https://archive-api.open-meteo.com/v1/archive"

url_params_dict ={
    "latitude": 27,
    "longitude": 30,
    "start_date": "2024-04-29",
    "end_date": "2025-04-30",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "rain",
        "precipitation",
        "cloud_cover",
        "wind_speed_10m",
    ],
    "timezone": "Africa/Cairo",
}

response = requests.get(api_data_url, params=url_params_dict)
if response.status_code == 200:
    data = response.json()
    print("success")

success


In [ ]:
import pandas as pd


timestamp = data["hourly"]["time"]
temperature_2m = data["hourly"]["temperature_2m"]
relative_humidity_2m = data["hourly"]["relative_humidity_2m"]
rain = data["hourly"]["rain"]
precipitation = data["hourly"]["precipitation"]
cloud_cover = data["hourly"]["cloud_cover"]
wind_speed_10m = data["hourly"]["wind_speed_10m"]

# combine all features into a single dataframe
data = pd.DataFrame({
    "timestamp": timestamp,
    "temperature_2m": temperature_2m,
    "relative_humidity_2m": relative_humidity_2m,
    "rain": rain,
    "precipitation": precipitation,
    "cloud_cover": cloud_cover,
    "wind_speed_10m": wind_speed_10m
})

8808

In [3]:
def split_data(df,train_size: float = 0.9):
    """
    Splits the data into training and testing sets.
    
    Args:
        df (pd.DataFrame): The DataFrame to split.
        train_size (float): The proportion of the data to include in the training set.
        
    Returns:
        tuple: A tuple containing the training and testing sets.
    """
    train_data = df[:int(len(df) * train_size)].copy()
    test_data = df[int(len(df) * train_size):].copy()
    return train_data, test_data

train, test = split_data(data, train_size=0.9)
def get_features_and_target(df: pd.DataFrame, target_col: str) -> (pd.DataFrame, pd.Series):
    """
    Splits the DataFrame into features and target variable.
    
    Args:
        df (pd.DataFrame): The DataFrame to split.
        target_col (str): The name of the target column.
        
    Returns:
        tuple: A tuple containing the features and target variable.
    """
    X = df.drop(columns=[target_col])
    y = df[target_col]
    return X, y

target_col = "temperature_2m"
X_train, y_train = get_features_and_target(train, target_col)
X_test, y_test = get_features_and_target(test, target_col)

# convert to numpy arrays
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

In [19]:
# take average of each feature per day
data["timestamp"] = pd.to_datetime(data["timestamp"])
data["day"] = data["timestamp"].dt.date
data = data.groupby("day").mean().reset_index()
data = data.drop(columns=["day"])

In [26]:
train_data = data[data.columns[0:2]]
train_data.set_index("timestamp", inplace=True)
train_data


,temperature_2m
timestamp,
2024-04-29 11:30:00,23.300000
2024-04-30 11:30:00,23.466667
2024-05-01 11:30:00,23.070833
2024-05-02 11:30:00,25.045833
2024-05-03 11:30:00,27.950000
...,...
2025-04-26 11:30:00,23.012500
2025-04-27 11:30:00,23.204167
2025-04-28 11:30:00,24.258333


In [32]:
from sktime.forecasting.fbprophet import Prophet

# Prepare the data for Prophet
# Initialize the Prophet model
model = Prophet(
    daily_seasonality=True,
)
model.fit(train_data)  # Fit the model with training data

# Fit the model

# Create a dataframe for future predictions
# Create a forecasting horizon (fh) as a range of integers
fh = pd.date_range(start=train_data.index[-1] + pd.Timedelta(days=1), periods=len(test), freq='D')

# Forecast the future values
forecast = model.predict(fh)

# Display the forecast
print(forecast)

23:11:57 - cmdstanpy - INFO - Chain [1] start processing
23:11:57 - cmdstanpy - INFO - Chain [1] done processing


                     temperature_2m
2025-05-01 11:30:00       25.611112
2025-05-02 11:30:00       25.963740
2025-05-03 11:30:00       25.905791
2025-05-04 11:30:00       25.960588
2025-05-05 11:30:00       26.174770
